<a href="https://colab.research.google.com/github/doyun1119/-/blob/main/Rogue(1980)_%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 제 1장 게임의 기본 구조

## 1-1 개발 환경 준비

In [2]:
import random
import os
import time

print("Rogue 개발환경을 준비 했습니다")
print("Python 버전:",__import__("sys").version.split()[0])

Rogue 개발환경을 준비 했습니다
Python 버전: 3.13.15


## 1-2 ASCII 던전 화면 만들기

In [5]:
#던전 크기
MAP_WIDTH = 40
MAP_HEIGHT = 20

#던전 타일
WALL = "#"
FLOOR = "."

#플레이어
PLAYER = "@"

#테스트용 던전
dungeon = [list("#"*MAP_WIDTH) for _ in range(MAP_HEIGHT)]

#가운데에 방 만들기
for y in range(2, MAP_HEIGHT - 2):
  for x in range(2, MAP_WIDTH - 2):
    dungeon[y][x] = FLOOR

#플레이어 위치
player_x = MAP_WIDTH // 2
player_y = MAP_HEIGHT // 2

#플레이어 표시
dungeon[player_y][player_x] = PLAYER

def draw_map():
  """현재 던전을 화면에 출력한다"""

  # Colab 화면을 어느정도 깔끔하게 유지
  print("\n" * 2)

  for row in dungeon:
    print("".join(row))


draw_map()




########################################
########################################
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##..................@.................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
########################################
########################################


## 1-3 플래이어 이동

In [6]:
def move_player(dx,dy):
  """
  플래이어를 dx,dy만큼 이동시킨다.
  이동할 수 있으면 True,
  이동할수 없으면 False를 반환한다.
  """

  global player_x,player_y

  new_x = player_x + dx
  new_y = player_y + dy

  #던전 범위를 벗어나는지 확인
  if new_x < 0 or new_x >= MAP_WIDTH:
        return False

  if new_y < 0 or new_y >= MAP_HEIGHT:
        return False

  # 벽인지 확인
  if dungeon[new_y][new_x] == WALL:
        return False

  # 기존 위치를 바닥으로 변경
  dungeon[player_y][player_x] = FLOOR

  # 플레이어 위치 변경
  player_x = new_x
  player_y = new_y

  # 새로운 위치에 플레이어 표시
  dungeon[player_y][player_x] = PLAYER

  return True


def process_input(command):
    """
    플레이어의 입력을 처리한다.
    """

    command = command.lower()

    if command == "w":
        return move_player(0, -1)

    elif command == "s":
        return move_player(0, 1)

    elif command == "a":
        return move_player(-1, 0)

    elif command == "d":
        return move_player(1, 0)

    return None

## 1-4 기본 게임 루프

In [9]:
def game_loop():
    """게임 전체를 실행한다."""

    print("=" * 40)
    print("         ROGUE")
    print("   ASCII Dungeon Adventure")
    print("=" * 40)

    print()
    print("W : 위")
    print("A : 왼쪽")
    print("S : 아래")
    print("D : 오른쪽")
    print("Q : 게임 종료")
    print()

    while True:

        # 화면 출력
        draw_map()

        # 현재 플레이어 위치
        print()
        print(f"위치: ({player_x}, {player_y})")

        # 명령 입력
        command = input("\n명령을 입력하세요: ")

        # 게임 종료
        if command.lower() == "q":
            print("\n게임을 종료합니다.")
            break

        # 이동 처리
        result = process_input(command)

        if result is True:
            print("이동했습니다.")

        elif result is False:
            print("벽 때문에 이동할 수 없습니다.")

        else:
            print("알 수 없는 명령입니다.")


# 게임 시작
game_loop()

         ROGUE
   ASCII Dungeon Adventure

W : 위
A : 왼쪽
S : 아래
D : 오른쪽
Q : 게임 종료




########################################
########################################
##...........@........................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
########################################
########################################

위치: (13, 2)

명령을 입력하세요: q

게임을 종료합니다.


# 제 2장 랜덤 던전 생성


## 2-1 방 생성

In [10]:
import random

# 던전 크기
MAP_WIDTH = 60
MAP_HEIGHT = 25

# 타일
WALL = "#"
FLOOR = "."

# 방 생성 설정
MIN_ROOM_WIDTH = 5
MAX_ROOM_WIDTH = 12

MIN_ROOM_HEIGHT = 4
MAX_ROOM_HEIGHT = 8

MAX_ROOMS = 8


class Room:
    """던전의 방을 나타내는 클래스"""

    def __init__(self, x, y, width, height):
        self.x = x
        self.y = y
        self.width = width
        self.height = height

    def center(self):
        """방의 중심 좌표를 반환한다."""
        center_x = self.x + self.width // 2
        center_y = self.y + self.height // 2

        return center_x, center_y

    def overlaps(self, other):
        """다른 방과 겹치는지 확인한다."""

        return (
            self.x <= other.x + other.width
            and self.x + self.width >= other.x
            and self.y <= other.y + other.height
            and self.y + self.height >= other.y
        )


def create_empty_dungeon():
    """벽으로 가득 찬 빈 던전을 만든다."""

    dungeon = []

    for y in range(MAP_HEIGHT):
        row = []

        for x in range(MAP_WIDTH):
            row.append(WALL)

        dungeon.append(row)

    return dungeon


def create_room(dungeon, room):
    """던전에 방을 만든다."""

    for y in range(room.y, room.y + room.height):
        for x in range(room.x, room.x + room.width):
            dungeon[y][x] = FLOOR


def generate_rooms():
    """랜덤한 방들을 생성한다."""

    rooms = []

    for _ in range(MAX_ROOMS):

        width = random.randint(
            MIN_ROOM_WIDTH,
            MAX_ROOM_WIDTH
        )

        height = random.randint(
            MIN_ROOM_HEIGHT,
            MAX_ROOM_HEIGHT
        )

        x = random.randint(
            1,
            MAP_WIDTH - width - 2
        )

        y = random.randint(
            1,
            MAP_HEIGHT - height - 2
        )

        new_room = Room(
            x,
            y,
            width,
            height
        )

        # 기존 방과 겹치는지 확인
        overlaps = False

        for room in rooms:
            if new_room.overlaps(room):
                overlaps = True
                break

        # 겹치지 않는 경우만 추가
        if not overlaps:
            rooms.append(new_room)

    return rooms


# 던전 생성
dungeon = create_empty_dungeon()

# 방 생성
rooms = generate_rooms()

# 던전에 방 표시
for room in rooms:
    create_room(dungeon, room)


# 던전 출력
for row in dungeon:
    print("".join(row))


print()
print("생성된 방:", len(rooms))

############################################################
############################################################
############################################################
############################################################
############################.....###########################
############################.....###########################
############################.....###########################
############################.....###########################
############################.....###########################
############################.....###########################
#####......#################.....###########################
#####......#################.....#######.........###########
#####......#############################.........###########
#####......#############################.........###########
#####......#############################.........###########
#############################......#####.........###########
########################

## 2-2 통로 연결

In [11]:
def create_horizontal_corridor(dungeon, x1, x2, y):
    """두 지점을 가로 통로로 연결한다."""

    start_x = min(x1, x2)
    end_x = max(x1, x2)

    for x in range(start_x, end_x + 1):
        dungeon[y][x] = FLOOR


def create_vertical_corridor(dungeon, y1, y2, x):
    """두 지점을 세로 통로로 연결한다."""

    start_y = min(y1, y2)
    end_y = max(y1, y2)

    for y in range(start_y, end_y + 1):
        dungeon[y][x] = FLOOR


def connect_rooms(dungeon, room1, room2):
    """두 방을 통로로 연결한다."""

    x1, y1 = room1.center()
    x2, y2 = room2.center()

    # 먼저 가로로 이동한 뒤 세로로 이동
    create_horizontal_corridor(
        dungeon,
        x1,
        x2,
        y1
    )

    create_vertical_corridor(
        dungeon,
        y1,
        y2,
        x2
    )


# 방들을 순서대로 연결
for i in range(1, len(rooms)):
    previous_room = rooms[i - 1]
    current_room = rooms[i]

    connect_rooms(
        dungeon,
        previous_room,
        current_room
    )


# 결과 출력
for row in dungeon:
    print("".join(row))

############################################################
############################################################
############################################################
############################################################
############################.....###########################
############################.....###########################
############################.....###########################
############################.....###########################
############################.................###############
############################.....###########.###############
#####......#################.....###########.###############
#####......#################.....#######.........###########
#####......###################.#########.........###########
#####......###################.#########.........###########
#####............................................###########
#############################......#####.........###########
########################

## 2-3 랜덤 던전 완성

In [12]:
def generate_dungeon():
  """완성된 랜덤 전전을 생성한다."""

  # 빈 던전 생성
  dungeon = create_empty_dungeon()

  # 방 생성
  rooms = generate_rooms()

  # 방을 던전에 추가
  for room in rooms:
    create_room(dungeon, room)

  # 방들을 통로로 연결
  for i in range(1, len(rooms)):
      connect_rooms(
          dungeon,
          rooms[i - 1],
          rooms[i]
      )

  return dungeon, rooms


# 새로운 던전 생성
dungeon, rooms = generate_dungeon()


# 던전 출력
for row in dungeon:
    print("".join(row))


print()
print("방의 개수:", len(rooms))

############################################################
############################################################
############################################################
###############################..........###################
######################.......##..........########.......####
######################...................########.......####
######################.......##..........########.......####
######################.......................####.......####
######################.......#######.#######.#######.#######
######################.......#######.#######.#######.#######
######################.......#######.#######.#######.#######
####################################.#######.#######.#######
########################...........#.#######.#######.#######
########################...........#.#######.#######.#######
########################...........#.####......#####.#######
########################.............####......#####.#######
########################

## 2-4 던전 검증

In [13]:
def get_floor_positions(dungeon):
    """던전에서 모든 바닥 위치를 찾는다."""

    positions = []

    for y in range(MAP_HEIGHT):
        for x in range(MAP_WIDTH):

            if dungeon[y][x] == FLOOR:
                positions.append((x, y))

    return positions


def flood_fill(dungeon, start_x, start_y):
    """시작점에서 이동 가능한 모든 바닥을 찾는다."""

    visited = set()
    stack = [(start_x, start_y)]

    directions = [
        (0, -1),  # 위
        (0, 1),   # 아래
        (-1, 0),  # 왼쪽
        (1, 0)    # 오른쪽
    ]

    while stack:

        x, y = stack.pop()

        # 이미 방문한 위치라면 무시
        if (x, y) in visited:
            continue

        # 범위를 벗어나면 무시
        if x < 0 or x >= MAP_WIDTH:
            continue

        if y < 0 or y >= MAP_HEIGHT:
            continue

        # 벽이면 이동할 수 없음
        if dungeon[y][x] != FLOOR:
            continue

        # 방문 처리
        visited.add((x, y))

        # 주변 위치 추가
        for dx, dy in directions:
            next_x = x + dx
            next_y = y + dy

            stack.append((next_x, next_y))

    return visited


def validate_dungeon(dungeon, rooms):
    """던전이 정상적으로 연결되어 있는지 검사한다."""

    if len(rooms) == 0:
        return False

    # 첫 번째 방의 중심에서 시작
    start_x, start_y = rooms[0].center()

    # 이동 가능한 모든 바닥 탐색
    reachable = flood_fill(
        dungeon,
        start_x,
        start_y
    )

    # 모든 방의 중심에 도달할 수 있는지 확인
    for room in rooms:

        x, y = room.center()

        if (x, y) not in reachable:
            return False

    return True


# 던전 검사
is_valid = validate_dungeon(
    dungeon,
    rooms
)

print()
print("던전 검증 결과:", is_valid)


던전 검증 결과: True


# 제 3장 플래이어 시스템

## 3-1 플래이어 능력치

## 3-2 이동 시스템 개선

## 3-3 상태창

## 3-4 플래이어 사망

# 제 4장 몬스터 시스템

## 4-1 몬스터 데이터

## 4-2 몬스터 배치

## 4-3 몬스터 이동

## 4-4 몬스터 종류

# 제 5장 전투 시스템

## 5-1 근접 공격

## 5-2 데미지 계산

## 5-3 몬스터 공격

## 5-4 경험치와 레벨업

# 제 6장 아이템 시스템

## 6-1 아이템 기본 시스템

## 6-2 무기

## 6-3 방어구

## 6-4 회복 아이템

## 6-5 인벤토리

# 제 7장 로그라이크 시스템

## 7-1 렌덤 아이템

## 7-2 렌덤 몬스터

## 7-3 렌덤 던전

## 7-4 영구 사망

## 7-5 새로운 게임

## 7-6 seed 시스템

# 제 8장 층 시스템


## 8-1 계단 생성

## 8-2 다음층 이동

## 8-3 층별 난이도 증가

## 8-4 몬스터 강화

## 8-5 아이템 변화

## 8-6 최종층

# 제 9장 최종목표와 엔딩

## 9-1 최종 던전

## 9-2 보스 몬스터

## 9-3 보스 전투

## 9-4 목표 아이템

## 9-5 엔딩

# 제 10장 UI와 최종 완성

## 10-1 화면 레이아웃

## 10-2 로그 메세지

## 10-3 명령어 안내

## 10-4 게임 시작 화면

## 10-5 게임 종료 화면

## 10-6 승리 화면

## 10-7 코드 정리

## 10-8 Colab 최종 통합
